In [2]:
import torch

def analyze(file_path, name="DATA"):
    data = torch.load(file_path)

    z = data["encodings"]
    y = data.get("labels")
    splits = data.get("split_ids")

    print(f"\n==== {name} ====")
    print("Shape:", z.shape)
    print("Dtype:", z.dtype)
    print("Min:", z.min().item())
    print("Max:", z.max().item())
    print("Mean:", z.mean().item())
    print("Std:", z.std().item())

    norms = torch.norm(z, dim=-1)
    print("Norm mean:", norms.mean().item())
    print("Norm std:", norms.std().item())

    if y is not None:
        print("Labels:", y.shape, "unique:", torch.unique(y))

    if splits is not None:
        print("Splits:", splits.shape)
        for i in torch.unique(splits):
            print(f"  split {i.item()} count:", (splits == i).sum().item())

    return z


def compare(old_path, new_path):
    old = analyze(old_path, "OLD")
    new = analyze(new_path, "NEW")

    print("\n==== COMPARISON ====")

    print("Shape match:", old.shape == new.shape)

    diff = (old - new).abs()
    print("Mean abs diff:", diff.mean().item())
    print("Max abs diff:", diff.max().item())

    cos = torch.nn.functional.cosine_similarity(
        old.view(-1, old.shape[-1]),
        new.view(-1, new.shape[-1]),
        dim=-1
    )
    print("Cosine similarity mean:", cos.mean().item())

    print("\n==== VERDICT GUIDE ====")
    print("cos > 0.99  → excellent")
    print("cos > 0.95  → acceptable")
    print("cos < 0.9   → investigate")


# -----------------------------
# USAGE
# -----------------------------
compare(
    "/home/nub/UvA/Master/DL2/spherical-flow-matching/sphere-encoder-main/workspace/experiments/sphere-small-small-cifar-10-32px/encoding1/encoded_dataset.pt",
    "/home/nub/UvA/Master/DL2/spherical-flow-matching/sphere-encoder-main/workspace/experiments/sphere-small-small-cifar-10-32px/encoding/encoded_dataset.pt"
)

/tmp/ipykernel_14682/3427346622.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path)



==== OLD ====
Shape: torch.Size([60000, 256, 4])
Dtype: torch.float16
Min: -1.9638671875
Max: 1.9541015625
Mean: -0.0029754638671875
Std: 0.5751953125
Norm mean: 1.0810546875
Norm std: 0.393310546875
Labels: torch.Size([60000]) unique: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
Splits: torch.Size([60000])
  split 0 count: 50000
  split 1 count: 10000

==== NEW ====
Shape: torch.Size([60000, 256, 4])
Dtype: torch.bfloat16
Min: -1.9765625
Max: 1.9609375
Mean: -0.0035858154296875
Std: 0.57421875
Norm mean: 1.078125
Norm std: 0.396484375
Labels: torch.Size([60000]) unique: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
Splits: torch.Size([60000])
  split 0 count: 50000
  split 1 count: 10000

==== COMPARISON ====
Shape match: True
Mean abs diff: 0.03397165238857269
Max abs diff: 0.25555419921875
Cosine similarity mean: 0.9949480891227722

==== VERDICT GUIDE ====
cos > 0.99  → excellent
cos > 0.95  → acceptable
cos < 0.9   → investigate
